# BERT

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.stem import PorterStemmer
import torch
from transformers import AutoTokenizer, AutoModel

In [5]:
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [7]:
target_title = "aspiring human resources"

inputs = tokenizer(
    target_title,
    return_tensors="pt",
    padding=True,
    truncation=True
)

print(inputs['input_ids'])
print(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

tensor([[  101, 22344,  2529,  4219,   102]])
['[CLS]', 'aspiring', 'human', 'resources', '[SEP]']


In [9]:
with torch.no_grad():
    outputs=model(**inputs)

token_embeddings = outputs.last_hidden_state

print(token_embeddings.shape)

torch.Size([1, 5, 768])


In [10]:
attention_mask = inputs['attention_mask'].unsqueeze(-1)

special_tokens_mask = torch.tensor(
    tokenizer.get_special_tokens_mask(
        inputs['input_ids'][0].tolist(),
        already_has_special_tokens=True
    )
).unsqueeze(0).unsqueeze(-1)

mask = attention_mask * (1 - special_tokens_mask)

sentence_embedding = (
    (token_embeddings * mask).sum(dim=1)
    / mask.sum(dim=1).clamp(min=1)
)

print(sentence_embedding.shape)

torch.Size([1, 768])
